In [ ]:
!pip install requests -quiet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import warnings
warnings.filterwarnings('ignore')
print("All libraries imported successfully")
print("Pandas version:",(pd.__version__))
print("Numpy version:",(np.__version__))
print("Requests version:",(requests.__version__))
df=pd.read_csv('messy_sales_data.csv')
print("Dataset loaded successfully")
print("Number of Rows:",df.shape[0])
print("Number of Columns:",df.shape[1])
print(df.columns.tolist())
# df.columns.tolist()
df.head(5)


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -u
All libraries imported successfully
Pandas version: 2.2.2
Numpy version: 2.0.2
Requests version: 2.32.4
Dataset loaded successfully
Number of Rows: 30
Number of Columns: 9
['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


OBJECTIVES:DAY3 -ETL
ETL-extract transforming load
ELT
HANDLING MISSING VALUES
REMOVE DUPLICATES
FIX DATATYPE ERROR
STANDARDIZE TEXT COLUMNS
AGGREGATE AND GRP CLEANED DATA USING GROUPBY AND AGG
API AND JSON
GET REQUEST
COMPLETE ETL PIPELINE:API->JSON->DATAFRAME->CLEAN->ANALYSE->VISULIZE

ETL- extract,transfor,load. 3 step process
EXTRACT-*pull data from source system.*source:apis,db,csv files,web scraping,iot sensors.*authentication(api key,token).*deal with pagination.*
store raw data exactly received(never modify source)
TRANSFORM-*handle miss value(null,blank,NaN).*
REMOVE DUP.*fix wrong data type(txt stored as num).*
standardize formats.*apply business rule.*filter invalid record.
LOAD-*write cleaned data to destination.*destination:sql db,data warehouse,csv,parquet.*choose load strategy.

ETL VS ELT:
ETL-use in cheap time,traditional approach,expensive for storage.
ELT-storage,cleaned data alone other delete(raw data for presed use),cloud.

clean messy_sales_data.csv
1.missing value in customer_name,quantity,category.
2.dup rows(order 1001/1005 are identical).
3.mixed data formats:YYYY-MM-DD and DD-MM-YYYY
4.inconsistant text case in customer_name(UPPER,lower,title)
5.wrong category value(keyboard labeled as electronics)

In [ ]:
print('='*55)
print('DATA QUALITY DIAGONOSIS REPORT')
print('='*55)
print("\n[1] MISSING VALUES PER COLUMN:")
print(df.isnull().sum())
print("\n[2] DUPLICATE ROWS:",df.duplicated().sum())
print('\n[3]DATA TYPES:')
print(df.dtypes)
print("\n[4]UNIQUE CATEGORIES:",df['category'].unique())
print('[4] Sample customer names:',df['customer_name'].dropna()[:8])
print('[4] Sample order_date values:',df['order_date'].unique()[:6])


DATA QUALITY DIAGONOSIS REPORT

[1] MISSING VALUES PER COLUMN:
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] DUPLICATE ROWS: 0

[3]DATA TYPES:
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4]UNIQUE CATEGORIES: ['Electronics' 'Accessories' nan]
[4] Sample customer names: 0    Ramesh Kumar
1      Priya Nair
2      AMIT VERMA
3    Sunita Patel
4    Ramesh Kumar
5     kiran mehta
6    Deepak Singh
8      Ananya Das
Name: customer_name, dtype: object
[4] Sample order_date values: ['2024-01-05' '2024-01-07' '2024-01-08' '2024-01-10' '07-01-2024'
 '2024-01-12']


In [ ]:
copy_df=df.copy()
#.copy() creates a copy of the existing data
print("Working copy created:",df.shape)

print(df.isnull().sum())
print("Before finding nulls:",df.isnull().sum().sum(),"total missing values")

#fix mixing customer_name with unknown user
unknown_name=df['customer_name'].fillna('Unknown User',inplace=True)

median_qty=df['quantity'].median()

df['quantity'].fillna(median_qty,inplace=True)
print("Filled missing quantity with median",median_qty)

#fix missing category
df['category'].fillna('Uncategorized',inplace=True)
print("After fixing nulls:",df.isnull().sum().sum(),"total missing values")

Working copy created: (30, 9)
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64
Before finding nulls: 7 total missing values
Filled missing quantity with median 2.0
After fixing nulls: 1 total missing values


In [ ]:
print(f'Before duplication:{len(df)}rows')
print(f'Duplicate rows:{df.duplicated().sum()}')
#show duplicates row
print('\nDuplicate rows:')
print(df[df.duplicated(keep=False)][['order_id','customer_name','product','order_date']])
#keep=false is all copies of dup
#this lets see both original and dup
#remove exact dup
df.drop_duplicates(inplace=True)
#remove row that are identical in all col
#first occurence is kept,subequent copies are remove
print(f'\nAfter deduplication:{len(df)}rows')
print(f'Rows removed:{len(df)-len(copy_df)}')

Before duplication:30rows
Duplicate rows:0

Duplicate rows:
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []

After deduplication:30rows
Rows removed:0


In [ ]:
print('Sample dates before parsing:')
print(df['order_date'].head(8).tolist())
#some are yyyy-mm-dd,some are dd-mm-yyyy
df['order_date']=pd.to_datetime(
    df['order_date'],
    dayfirst=False,#try yyyy-mm-dd format first
    errors='coerce'#if parsing fails,put NaT(not a time)instead of crashing
)
#pd.to_datetime() converts strings to proper deleting objects
#dayfirst=False: treats yyyy-mm-dd as primary format
#check for any unparseable dates
nat_count=df['order_date'].isnull().sum()
print(f'\n Unparseable dates(NaT):{nat_count}')
#extract date components
df['year']=df['order_date'].dt.year
df['month']=df['order_date'].dt.month
df['month_name']=df['order_date'].dt.strftime('%B')
#.dt is datetime accessor give year,month,day
#.strftime('%B')return full month name:'jan','feb',etc
print('\n Sample dates after parsing')
print(df[['order_date','year','month','month_name']].head(5))

Sample dates before parsing:
['2024-01-05', '2024-01-07', '2024-01-08', '2024-01-10', '2024-01-05', '07-01-2024', '2024-01-12', '2024-01-13']

 Unparseable dates(NaT):2

 Sample dates after parsing
  order_date    year  month month_name
0 2024-01-05  2024.0    1.0    January
1 2024-01-07  2024.0    1.0    January
2 2024-01-08  2024.0    1.0    January
3 2024-01-10  2024.0    1.0    January
4 2024-01-05  2024.0    1.0    January


In [ ]:
print('Before standardization:',df['customer_name'].unique()[:6])
df['cutomer_name']=(
    df['customer_name']
    .str.strip()#remove leading/trailing whitespace
    .str.title()#convert to title case:'amit varma'-> 'Amit Varma'
)
#.str is pandas string accessor apply string name method
#.strip() remove space at edges
#.title() capializes first letter
print('After standardization:',df['customer_name'].unique()[:6])
print(f'\nBefore:keyboard rows with electronis category')
# wrong_maskis the boolean series
#& is bitwise AND operator
wrong_mask=(df['product']=='keyboard')&(df['category']=='electronics')
print(df[wrong_mask][['product','category']])
df.loc[wrong_mask,'category']='Accessories'
#df.loc[row_mask,col_name]=value
#this set'category' to 'accessories' only for rows whrere wrong_mask is true
#all other rows are unaffected
print('After fix:unique categories:',df['category'].unique())

Before standardization: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']
After standardization: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']

Before:keyboard rows with electronis category
Empty DataFrame
Columns: [product, category]
Index: []
After fix:unique categories: ['Electronics' 'Accessories' 'Uncategorized']


In [ ]:
df['quantity']=pd.to_numeric(df['quantity'],errors='coerce').astype(int)
df['unit_price']=pd.to_numeric(df['unit_price'],errors='coerce')
#pd.to_numeric covert any col to numeric type
#errors='coerce' turns unconvertible values to NaN instead of crashing
#.astype(int) converts float(2.0)to integer(2) for quality
#create revenue col
df['revenue']=df['quantity']*df['unit_price']
print('Revenuecol created:')
print(df[['customer_name','product','quantity','unit_price','revenue']].head(5))
print(f'\n Total revenue across all orders:{df["revenue"].sum():,.0f}')

Revenuecol created:
  customer_name   product  quantity  unit_price  revenue
0  Ramesh Kumar    Laptop         2       45000    90000
1    Priya Nair       NaN         1       15000    15000
2    AMIT VERMA  Keyboard         3        1200     3600
3  Sunita Patel   Monitor         2       22000    44000
4  Ramesh Kumar    Laptop         2       45000    90000

 Total revenue across all orders:818,000


In [ ]:
print('='*55)
print('post-cleaning validation report')
print('='*55)
print(f'original rows:{len(df)}')
print(f'cleaned rows:{len(copy_df)}')
print(f'rows removed:{len(df)-len(copy_df)}(duplicates)')
print(f'missing values:{df.isnull().sum().sum()}')
print(f'duplicates:{df.duplicated().sum()}')
print(f'date nulls:{df["order_date"].isnull().sum()}')
print(f'revenue NaN:{df["revenue"].isnull().sum()}')
print(f'categories:{sorted(df["category"].unique())}')
print('='*55)
all_clean=(
    df.isnull().sum().sum()==0 and
    df.duplicated().sum()==0
)
print(f'Data is clean:{all_clean}')

post-cleaning validation report
original rows:30
cleaned rows:30
rows removed:0(duplicates)
missing values:9
duplicates:0
date nulls:2
revenue NaN:0
categories:['Accessories', 'Electronics', 'Uncategorized']
Data is clean:False


In [ ]:
product_rev=(
    df.groupby('product')['revenue']
    .sum()
    .reset_index()
    .sort_values('revenue',ascending=False)
)
print('Revenue by product:')
print(product_rev.to_string(index=False))
category_summary=df.groupby('category').agg(
    total_revenue=('revenue','sum'),
    avg_order_value=('revenue','mean'),
    num_orders=('order_id','count'),
    unique_products=('product','nunique')
).round(2).reset_index()
print('\nCategory summary:')
print(category_summary.to_string(index=False))

Revenue by product:
   product  revenue
    Laptop   540000
   Monitor   154000
Headphones    28000
     Mouse    20800
  Keyboard    20400
    Webcam    20000
   USB Hub    19800

Category summary:
     category  total_revenue  avg_order_value  num_orders  unique_products
  Accessories          76200          5861.54          13                4
  Electronics         697800         43612.50          16                4
Uncategorized          44000         44000.00           1                1


In [ ]:
df.to_csv('clean_sales_data.csv',index=False)

print('cleaned data saved to: clean_sales_data.csv')
print(f'final dataset:{df.shape[0]}rows x{df.shape[1]} columns')
print('\nETL Pipeline for Sales Data: COMPLETE')
print(' EXTRACT -> messy_sales_data.csv loaded')
print(' TRANSFORM -> messy_sales_data.csv cleaned')
print(' LOAD -> clean_sales_data.csv saved')

cleaned data saved to: clean_sales_data.csv
final dataset:30rows x14 columns

ETL Pipeline for Sales Data: COMPLETE
 EXTRACT -> messy_sales_data.csv loaded
 TRANSFORM -> messy_sales_data.csv cleaned
 LOAD -> clean_sales_data.csv saved


In [ ]:
API_KEY='0f01a777b6056b52455e09ef1e905c63'
BASE_URL='https://api.openweathermap.org/data/2.5/weather'
CITIES=['Mumbai','Delhi','Banglore','chennai','Hydrabad','kolkata','Pune','Jaipur']
print(f'api configured for {len(CITIES)}cities')
print(f'cities:{CITIES}')
print('\nIMPORTANT:Replace YOUR_API_KEY with your own API key')

api configured for 8cities
cities:['Mumbai', 'Delhi', 'Banglore', 'chennai', 'Hydrabad', 'kolkata', 'Pune', 'Jaipur']

IMPORTANT:Replace YOUR_API_KEY with your own API key


In [1]:
import requests
import pandas as pd

# --- EXTRACT --- #

API_KEY = '8ce86da271c6e3a978c2359e88324349' # Replace with your actual OpenWeatherMap API key
BASE_URL = 'http://api.openweathermap.org/data/2.5/weather'
CITIES = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad', 'Kolkata', 'Pune', 'Jaipur']

def fetch_weather(city, API_KEY):
    """
    Fetch current weather data for a given city.
    Returns a dictionary with weather metrics, or None on failure.
    """
    params = {
        'q':     city,
        'appid': API_KEY,
        'units': 'metric'
    }
    try:
        response = requests.get(BASE_URL, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            return {
                'city':        city,
                'temperature': round(data['main']['temp'], 1),
                'feels_like':  round(data['main']['feels_like'], 1),
                'humidity':    data['main']['humidity'],
                'pressure':    data['main']['pressure'],
                'wind_speed':  data['wind']['speed'],
                'condition':   data['weather'][0]['description'].title(),
                'visibility':  data.get('visibility', 0) // 1000
            }
        else:
            print(f'  ERROR {response.status_code} for {city}: {response.json().get("message","unknown error")}')
            return None

    except requests.exceptions.ConnectionError:
        print(f'  CONNECTION ERROR for {city} — check internet connection')
        return None
    except requests.exceptions.Timeout:
        print(f'  TIMEOUT for {city} — API did not respond in 10 seconds')
        return None


print('Calling Weather API...')
weather_records = []
for city in CITIES:
    print(f'  Fetching: {city}...', end='')
    record = fetch_weather(city, API_KEY)
    if record:
        weather_records.append(record)
        print(f' {record["temperature"]}°C, {record["condition"]}')
    else:
        print(' FAILED')

print(f'\nSuccessfully fetched: {len(weather_records)}/{len(CITIES)} cities')

# --- TRANSFORM --- #

# Create DataFrame from API results
weather_df = pd.DataFrame(weather_records)

print('\nWeather DataFrame created and data cleaned (structured):')
print(weather_df.head())
print(f'Shape: {weather_df.shape}')
print(f'Missing values: {weather_df.isnull().sum().sum()}')

# --- LOAD --- #

# Save the DataFrame to a CSV file
output_filename = 'weather_data_etl_output.csv'
weather_df.to_csv(output_filename, index=False)

print(f'\nWeather data saved to: {output_filename}')
print('\nWeather Data ETL Pipeline: COMPLETE')
print('  EXTRACT   → OpenWeatherMap API called')
print('  TRANSFORM → JSON parsed, DataFrame built, units converted')
print(f'  LOAD      → {output_filename} saved')

Calling Weather API...
  Fetching: Mumbai... 32.0°C, Haze
  Fetching: Delhi... 28.1°C, Haze
  Fetching: Bangalore... 25.3°C, Scattered Clouds
  Fetching: Chennai... 29.0°C, Mist
  Fetching: Hyderabad... 30.2°C, Haze
  Fetching: Kolkata... 31.0°C, Haze
  Fetching: Pune... 28.1°C, Clear Sky
  Fetching: Jaipur... 30.6°C, Haze

Successfully fetched: 8/8 cities

Weather DataFrame created and data cleaned (structured):
        city  temperature  feels_like  humidity  pressure  wind_speed  \
0     Mumbai         32.0        39.0        66      1011        5.14   
1      Delhi         28.1        29.2        57      1006        4.12   
2  Bangalore         25.3        26.0        81      1015        8.05   
3    Chennai         29.0        35.6        84      1006        7.20   
4  Hyderabad         30.2        32.9        58      1013        6.17   

          condition  visibility  
0              Haze           4  
1              Haze           4  
2  Scattered Clouds           6  
3       

In [2]:
import pandas as pd

# --- CONFIGURATION --- #
excel_file_path = 'employee_salary_data.xlsx' # Make sure this Excel file is uploaded or accessible
output_csv_file = 'cleaned_employee_salary.csv'

# --- EXTRACT --- #
print('--- Starting Employee Salary ETL Project ---')

try:
    # Attempt to load data from Excel
    df_raw = pd.read_excel(excel_file_path)
    print(f'Successfully loaded data from {excel_file_path}. Shape: {df_raw.shape}')
except FileNotFoundError:
    print(f'Warning: {excel_file_path} not found. Creating dummy data for demonstration.')
    # Create a dummy DataFrame if the Excel file is not found
    data = {
        'Employee ID': [101, 102, 103, 104, 105, 101, 106, 107, 108],
        'Name': ['Alice Smith', 'Bob Johnson', 'Charlie Brown', 'Diana Prince', 'Eve Adams', 'Alice Smith', 'Frank White', 'Grace Black', 'Harry Green'],
        'Department': ['HR', 'IT', 'Finance', 'Marketing', 'HR', 'HR', 'IT', 'Finance', 'Marketing'],
        'Monthly Salary': [5000, 6000, 7500, 5500, 5200, 5000, 6200, 7800, 5800]
    }
    df_raw = pd.DataFrame(data)
    print(f'Dummy DataFrame created. Shape: {df_raw.shape}')

df = df_raw.copy() # Create a working copy

# --- TRANSFORM --- #
print('\n--- Data Transformation ---')

# 1. Remove duplicate records
print(f'Before dropping duplicates: {len(df)} rows')
duplicates_before = df.duplicated().sum()
df.drop_duplicates(inplace=True)
print(f'Removed {duplicates_before} duplicate rows. After dropping duplicates: {len(df)} rows')

# 2. Calculate Yearly Salary
if 'Monthly Salary' in df.columns:
    df['Yearly Salary'] = df['Monthly Salary'] * 12
    print('Calculated "Yearly Salary" based on "Monthly Salary".')
elif 'salary' in df.columns:
    df['Yearly Salary'] = df['salary'] * 12
    print('Calculated "Yearly Salary" based on "salary".')
else:
    print('Could not find "Monthly Salary" or "salary" column to calculate yearly salary.')

print('\nTransformed Data (first 5 rows):')
print(df.head())
print(f'Final DataFrame shape after transformations: {df.shape}')
print(f'Missing values after transformations: {df.isnull().sum().sum()}')

# --- LOAD --- #
print('\n--- Data Loading ---')
df.to_csv(output_csv_file, index=False)
print(f'Cleaned employee salary data saved to: {output_csv_file}')

print('\n--- Employee Salary ETL Project: COMPLETE ---')
print('  EXTRACT   → Data loaded from Excel (or dummy data generated)')
print('  TRANSFORM → Duplicates removed, Yearly Salary calculated')
print(f'  LOAD      → {output_csv_file} saved')

--- Starting Employee Salary ETL Project ---
Dummy DataFrame created. Shape: (9, 4)

--- Data Transformation ---
Before dropping duplicates: 9 rows
Removed 1 duplicate rows. After dropping duplicates: 8 rows
Calculated "Yearly Salary" based on "Monthly Salary".

Transformed Data (first 5 rows):
   Employee ID           Name Department  Monthly Salary  Yearly Salary
0          101    Alice Smith         HR            5000          60000
1          102    Bob Johnson         IT            6000          72000
2          103  Charlie Brown    Finance            7500          90000
3          104   Diana Prince  Marketing            5500          66000
4          105      Eve Adams         HR            5200          62400
Final DataFrame shape after transformations: (8, 5)
Missing values after transformations: 0

--- Data Loading ---
Cleaned employee salary data saved to: cleaned_employee_salary.csv

--- Employee Salary ETL Project: COMPLETE ---
  EXTRACT   → Data loaded from Excel (or dum